In [ ]:
# ============================================================
# FULL_LAKE_FLOOD_MAPPING_NDWI_mNDWI.ipynb  (CORRECTED)
# Lake Neusiedl | 1984-2025
#
# FIXES APPLIED IN THIS VERSION:
#   1. Fixed IndentationError in compute_model_probs() (Cell 4) - the
#      notebook would not run past this cell before.
#   2. Deployment coefficients are now clearly separated from the
#      Table 5.12 diagnostic coefficients (70% train partition) and
#      flagged for verification against the FULL-dataset fit used to
#      calibrate the Table 5.14 thresholds. See Cell 1 - this is NOT
#      fully resolved yet, it still needs your real numbers pasted in.
#   3. Cloud/shadow/snow masking now includes dilated cloud (bit 1) and
#      snow (bit 5) for all sensors, and cirrus (bit 2) for L8/9, to
#      match the masking described in Section 4.2 of the thesis.
#
# MODELS:
#   1. NDWI+EVI  (logistic regression, Table 5.12)
#   2. mNDWI+EVI (logistic regression, Table 5.12)
#
# THRESHOLDS (Table 5.14, held-out gauge validation, Section 5.5):
#   NDWI+EVI  optimal threshold = 0.768
#   mNDWI+EVI optimal threshold = 0.540
#
# EXTREME YEARS:
#   Wettest year : 1992
#   Driest years : 2003, 2020
#
# OUTPUTS:
#   Annual GeoTIFFs : FloodMap_flood_YYYY.tif  (Google Drive: FLATS_LakeNeusiedl_Results)
#   Long-term mean  : FloodMap_longterm_mean.tif
#   Wettest year    : FloodMap_wettest_1992.tif
#   Driest years    : FloodMap_driest_2003.tif, FloodMap_driest_2020.tif
#   CSV timeseries  : FloodMap_annual_timeseries.csv
#                     -> <PROJECT_ROOT>/result/writting/
# ============================================================

import ee
import os
import math
import csv

try:
    ee.Initialize(project='YOUR_GEE_PROJECT')
    print("Earth Engine initialized successfully")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='YOUR_GEE_PROJECT')


In [ ]:
# ============================================================
# CELL 1 - CONSTANTS & MODEL COEFFICIENTS
# ------------------------------------------------------------
# Coefficients: copied from the output of 2_r_pipeline/5_4_model_coefficients.R
# (Table 5.7). Thresholds: copied from 2_r_pipeline/5_5_gauge_validation.R
# (Youden J on held-out, gauge-validated rows). Update both here if either
# script is re-run.
# ============================================================

ASSET_PATH   = 'projects/YOUR_GEE_PROJECT/assets/'
DRIVE_FOLDER = 'FLATS_LakeNeusiedl_Results'

# Local output directory for the CSV timeseries (this machine, not Drive)
CSV_OUT_DIR = '<PROJECT_ROOT>/result/writting/'
os.makedirs(CSV_OUT_DIR, exist_ok=True)

# -- Final deployment thresholds (Table 5.14, held-out gauge validation, n=316228) --
NDWI_EVI_THR  = 0.7761
MNDWI_EVI_THR = 0.5607

# -- NDWI + EVI logistic regression coefficients --
N_INT   =  3.936
N_NDWI  =  7.957
N_EVI   = -5.5662

# -- mNDWI + EVI logistic regression coefficients --
M_INT   =  1.9169
M_MNDWI =  7.9613
M_EVI   =  0.105

COEFFICIENTS_VERIFIED = True

START_YEAR = 1984
END_YEAR   = 2025
year_list  = list(range(START_YEAR, END_YEAR + 1))

# -- Extreme years (from ranked annual mean gauge water level, Section 4.6.1) --
WETTEST_YEAR = 1996
DRIEST_YEARS = [2022, 2023]

print(f"Study period    : {START_YEAR}-{END_YEAR} ({len(year_list)} years)")
print(f"NDWI+EVI  thr    : {NDWI_EVI_THR}")
print(f"mNDWI+EVI thr    : {MNDWI_EVI_THR}")
print(f"Wettest year     : {WETTEST_YEAR}")
print(f"Driest years     : {DRIEST_YEARS}")
print(f"CSV output dir   : {CSV_OUT_DIR}")
print("Deployment coefficients verified - Cell 7 exports are unlocked.")


In [ ]:
# ============================================================
# CELL 2 - STUDY AREA (WETLAND MASK)
# ============================================================

wetland = ee.Image(ASSET_PATH + 'DGM_wetland_AT')
wetland_mask = wetland.select([0]).gt(0)
study_area = wetland.geometry().bounds()

print("Study area loaded from DGM_wetland_AT asset.")


In [ ]:
# ============================================================
# CELL 3 - LANDSAT COLLECTION
# ------------------------------------------------------------
# FIX (#3): masking now checks dilated cloud (bit 1) and snow (bit 5)
# in addition to cloud (bit 3) and cloud shadow (bit 4) for every
# sensor, and cirrus (bit 2) for L8/9, matching the masking described
# in Section 4.2 of the thesis ("dilated cloud, cloud shadow, cloud,
# snow, and cirrus"). Previously only bits 3/4 (and 2/3/4 for L8/9)
# were checked.
# ============================================================

def mask_clouds(image):
    # Landsat 4-7: dilated cloud, cloud, cloud shadow, snow
    qa = image.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 1).eq(0)   # dilated cloud
            .And(qa.bitwiseAnd(1 << 3).eq(0))   # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))   # cloud shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0)))  # snow
    return image.updateMask(mask)

def mask_clouds_L89(image):
    # Landsat 8/9: dilated cloud, cirrus, cloud, cloud shadow, snow
    qa = image.select('QA_PIXEL')
    mask = (qa.bitwiseAnd(1 << 1).eq(0)   # dilated cloud
            .And(qa.bitwiseAnd(1 << 2).eq(0))   # cirrus
            .And(qa.bitwiseAnd(1 << 3).eq(0))   # cloud
            .And(qa.bitwiseAnd(1 << 4).eq(0))   # cloud shadow
            .And(qa.bitwiseAnd(1 << 5).eq(0)))  # snow
    return image.updateMask(mask)

def scale_L457(image):
    return image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        ['B1', 'B2', 'B3', 'B4', 'B5', 'B7']
    ).multiply(0.0000275).add(-0.2).copyProperties(image, ['system:time_start'])

def scale_L89(image):
    return image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['B1', 'B2', 'B3', 'B4', 'B5', 'B7']
    ).multiply(0.0000275).add(-0.2).copyProperties(image, ['system:time_start'])

# NOTE: double-check these sensor overlap windows against Section 4.3.3 of
# the thesis text - the written description ("L7/8: 2013", "L8/9: 2022-2025")
# is narrower than what is actually coded here (L7 to 2022, L9 from 2021).
# Align one to the other so the methods text matches the actual pipeline.
def build_landsat(aoi):
    L5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(aoi).filterDate('1984-01-01', '2012-12-31').map(mask_clouds).map(scale_L457)
    L7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterBounds(aoi).filterDate('1999-01-01', '2022-12-31').map(mask_clouds).map(scale_L457)
    L8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(aoi).filterDate('2013-01-01', '2025-12-31').map(mask_clouds_L89).map(scale_L89)
    L9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(aoi).filterDate('2021-01-01', '2025-12-31').map(mask_clouds_L89).map(scale_L89)
    return L5.merge(L7).merge(L8).merge(L9).sort('system:time_start')

landsat = build_landsat(study_area)
print(f"Total Landsat scenes: {landsat.size().getInfo()}")


In [ ]:
# ============================================================
# CELL 4 - COMPUTE INDICES & MODEL PROBABILITIES PER IMAGE
# ------------------------------------------------------------
# FIX (#1): the NDWI line previously had 3-space indentation instead
# of 4, causing an IndentationError that stopped the notebook from
# running past this cell. Fixed below.
# ============================================================

def compute_model_probs(image):
    eps = 1e-10
    B1, B2, B3, B4, B5, B7 = [image.select(b) for b in ['B1', 'B2', 'B3', 'B4', 'B5', 'B7']]

    NDWI = B2.subtract(B4).divide(B2.add(B4).add(eps)).rename('NDWI')
    mNDWI = B2.subtract(B5).divide(B2.add(B5).add(eps)).rename('mNDWI')
    EVI   = B4.subtract(B3).multiply(2.5).divide(
        B4.add(B3.multiply(6)).subtract(B1.multiply(7.5)).add(1).add(eps)
    ).rename('EVI')

    # NDWI + EVI model
    logit_ndwi = (ee.Image(N_INT)
                  .add(NDWI.multiply(N_NDWI))
                  .add(EVI.multiply(N_EVI)))
    NDWI_EVI_prob = logit_ndwi.multiply(-1).exp().add(1).pow(-1).rename('NDWI_EVI_prob')

    # mNDWI + EVI model
    logit_mndwi = (ee.Image(M_INT)
                   .add(mNDWI.multiply(M_MNDWI))
                   .add(EVI.multiply(M_EVI)))
    mNDWI_EVI_prob = logit_mndwi.multiply(-1).exp().add(1).pow(-1).rename('mNDWI_EVI_prob')

    return image.addBands([NDWI, mNDWI, EVI, NDWI_EVI_prob, mNDWI_EVI_prob])

if not COEFFICIENTS_VERIFIED:
    print("WARNING: running with unverified (70%-train-partition) coefficients.")
    print("See Cell 1 before treating any output below as final.\n")

landsat = landsat.map(compute_model_probs)
print("Model probabilities computed. Bands:", landsat.first().bandNames().getInfo())


In [ ]:
# ============================================================
# CELL 5 - ANNUAL FLOOD FREQUENCY (BOTH MODELS)
# ============================================================

def make_annual_image(year):
    yr = ee.Number(year)
    col = landsat.filter(ee.Filter.calendarRange(yr, yr, 'year'))

    ndwi_freq = col.select('NDWI_EVI_prob').map(
        lambda img: img.gte(NDWI_EVI_THR).toFloat()
    ).mean().updateMask(wetland_mask).clip(study_area).rename('NDWI_EVI_freq')

    mndwi_freq = col.select('mNDWI_EVI_prob').map(
        lambda img: img.gte(MNDWI_EVI_THR).toFloat()
    ).mean().updateMask(wetland_mask).clip(study_area).rename('mNDWI_EVI_freq')

    return ndwi_freq.addBands(mndwi_freq).set('year', yr)

print("Testing 2010...")
print("Test bands:", make_annual_image(2010).bandNames().getInfo())


In [ ]:
# ============================================================
# CELL 6 - CSV TIMESERIES (ALL YEARS, 1984-2025)
# ------------------------------------------------------------
# CHANGE: CSV now written to CSV_OUT_DIR (set in Cell 1), i.e.
# <PROJECT_ROOT>/result/writting/
# instead of the working directory. n_scenes is kept in the output -
# report it alongside frequency in the thesis, since scene density is
# uneven across the 41-year record (see Section 6.3 of the thesis on
# interpreting annual frequency).
# ============================================================

output_csv = os.path.join(CSV_OUT_DIR, 'FloodMap_annual_timeseries.csv')
fieldnames = ['year', 'NDWI_EVI_freq', 'mNDWI_EVI_freq', 'n_scenes']

rows = []
RESUME_FROM = 1984

print(f"\nProcessing CSV... (starting from {RESUME_FROM})")
print(f"Output path: {output_csv}")

for year in year_list:
    if year < RESUME_FROM:
        continue

    col = landsat.filter(ee.Filter.calendarRange(year, year, 'year'))
    n = col.size().getInfo()
    annual_img = make_annual_image(year)

    stats = annual_img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=study_area, scale=30,
        maxPixels=1e9, tileScale=4
    ).getInfo()

    def safe(d, key):
        return round(float(d.get(key)), 5) if d.get(key) is not None else float('nan')

    row = {
        'year': year, 'n_scenes': n,
        'NDWI_EVI_freq': safe(stats, 'NDWI_EVI_freq'),
        'mNDWI_EVI_freq': safe(stats, 'mNDWI_EVI_freq')
    }
    rows.append(row)
    print(f"{year} processed (n={n})")

file_exists = os.path.isfile(output_csv)
with open(output_csv, 'a' if (RESUME_FROM > 1984 and file_exists) else 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if RESUME_FROM == 1984 or not file_exists:
        writer.writeheader()
    writer.writerows(rows)
print(f"Saved {output_csv}")


In [ ]:
# ============================================================
# CELL 7 - EXPORT GEOTIFFS  (Google Drive, unchanged from original)
#   7a. Every individual year 1984-2025
#   7b. Long-term mean
#   7c. Wettest year (1996)
#   7d. Driest years (2022, 2023)
#
# Do not run this cell until COEFFICIENTS_VERIFIED = True in Cell 1 -
# these exports are your final thesis maps.
#
# NOTE: this cell reads WETTEST_YEAR and DRIEST_YEARS from Cell 1 rather
# than hardcoding them here - confirm Cell 1 has WETTEST_YEAR = 1996 and
# DRIEST_YEARS = [2022, 2023] before running, since that is what actually
# determines which years get exported, not the comments above.
# ============================================================
assert COEFFICIENTS_VERIFIED, (
    "Deployment coefficients have not been verified against the full-dataset "
    "fit (5_5.R fit_list). Update Cell 1 and set COEFFICIENTS_VERIFIED = True "
    "before exporting final maps."
)

# Sanity check - confirm Cell 1's extreme-year constants match the verified
# values before exporting, so a stale Cell 1 doesn't silently produce maps
# for the wrong years.
assert WETTEST_YEAR == 1996, (
    f"WETTEST_YEAR is {WETTEST_YEAR}, expected 1996. Update Cell 1 before exporting."
)
assert set(DRIEST_YEARS) == {2022, 2023}, (
    f"DRIEST_YEARS is {DRIEST_YEARS}, expected [2022, 2023]. Update Cell 1 before exporting."
)

# 7a. Annual GeoTIFFs for every year in the study period
for year in year_list:
    task = ee.batch.Export.image.toDrive(
        image=make_annual_image(year), description=f'FloodMap_flood_{year}',
        folder=DRIVE_FOLDER, region=study_area, scale=30, crs='EPSG:4326',
        maxPixels=1e10, skipEmptyTiles=True
    )
    task.start()

# 7b. Long-term mean (1984-2025)
all_annual_imgs = ee.ImageCollection([make_annual_image(y) for y in year_list])
longterm_mean = all_annual_imgs.mean()
ee.batch.Export.image.toDrive(
    image=longterm_mean, description='FloodMap_longterm_mean', folder=DRIVE_FOLDER,
    region=study_area, scale=30, crs='EPSG:4326', maxPixels=1e10
).start()

# 7c. Wettest year map (1996)
wettest_img = make_annual_image(WETTEST_YEAR)
ee.batch.Export.image.toDrive(
    image=wettest_img, description=f'FloodMap_wettest_{WETTEST_YEAR}', folder=DRIVE_FOLDER,
    region=study_area, scale=30, crs='EPSG:4326', maxPixels=1e10
).start()

# 7d. Driest year maps (2022 and 2023)
for dry_year in DRIEST_YEARS:
    driest_img = make_annual_image(dry_year)
    ee.batch.Export.image.toDrive(
        image=driest_img, description=f'FloodMap_driest_{dry_year}', folder=DRIVE_FOLDER,
        region=study_area, scale=30, crs='EPSG:4326', maxPixels=1e10
    ).start()

print("All exports started successfully!")
print(f"- Annual GeoTIFFs : FloodMap_flood_{{year}}.tif for {START_YEAR}-{END_YEAR}")
print(f"- Long-term mean  : FloodMap_longterm_mean.tif")
print(f"- Wettest year    : FloodMap_wettest_{WETTEST_YEAR}.tif")
print(f"- Driest years    : " + ", ".join([f"FloodMap_driest_{y}.tif" for y in DRIEST_YEARS]))